# Cross-Dataset Generalization -- FEVER

Runs the zero-shot NLI methods (trained on nothing, so "cross-dataset" here means *evaluated* on a different distribution) on FEVER claim/evidence pairs, reusing the exact same detectors from `01_zero_shot_nli.ipynb`. This is the evaluation planned in `docs/proposal.pdf` Section 4 and flagged as incomplete in `docs/report.pdf` Section 5.4.

Uses `pietrolesci/nli_fever` -- a pre-converted, script-free mirror of FEVER -- since the original `fever` dataset repo relies on a loading script that current `datasets` versions no longer execute. REFUTES -> hallucinated (1), SUPPORTS -> faithful (0); NOT ENOUGH INFO examples are dropped (no HaluEval analogue).

In [ ]:
import sys, json
sys.path.insert(0, '..')

from src.data.loader import load_fever
from src.models.zero_shot import run_zero_shot_nli
from src.evaluation.metrics import compute_metrics, print_report

fever_ds = load_fever(split='dev', num_samples=2000)
print(f'Loaded {len(fever_ds)} FEVER dev examples (SUPPORTS/REFUTES only)')

In [ ]:
bart_fever = run_zero_shot_nli(fever_ds, model_name='facebook/bart-large-mnli')
print_report(bart_fever.labels, bart_fever.predictions, title='BART-MNLI on FEVER (cross-domain)')

In [ ]:
with open('../outputs/tables/fever_cross_domain_results.json', 'w') as f:
    json.dump({
        'BART-MNLI on FEVER': compute_metrics(bart_fever.labels, bart_fever.predictions).as_dict(),
    }, f, indent=2)